In [2]:
from pyspark.sql import SparkSession
import getpass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_RDD") \
    .getOrCreate()

# Lấy SparkContext từ SparkSession để dùng RDD
sc = spark.sparkContext

print("Spark version:", sc.version)

Spark version: 4.0.1


# **Bài 6: Phân Tích Đánh Giá Theo Thời Gian**

In [11]:
movies_rdd = sc.textFile("data/movies.txt")
ratings_rdd = sc.textFile("data/ratings_*.txt")
user_rdd = sc.textFile("data/users.txt")
occupation_rdd = sc.textFile("data/occupations.txt")

In [12]:
ratings_rdd.take(5)

['7,1020,4.5,1577836800',
 '23,1015,3.5,1577923200',
 '45,1030,4.0,1578009600',
 '12,1047,3.0,1578096000',
 '38,1012,4.5,1578182400']

Do timestamp là số giây kể từ ngày 1 tháng 1 năm 1970 nên ta sẽ đổi cột timestamp ra năm

In [ ]:
import datetime
def calculate_year(timestamp):
    return datetime.datetime.fromtimestamp(int(timestamp)).year

ratings_rdd = ratings_rdd.map(lambda x : x.split(",")).map(lambda x : (x[0], x[1], x[2], calculate_year(x[3])))
ratings_rdd.take(5)



[('7', '1020', '4.5', 2020),
 ('23', '1015', '3.5', 2020),
 ('45', '1030', '4.0', 2020),
 ('12', '1047', '3.0', 2020),
 ('38', '1012', '4.5', 2020)]

In [20]:
#Map theo nam
ratingsNew = ratings_rdd.map(lambda x: (x[3], (float(x[2]), 1)))
ratingsNew.take(5)


[(2020, (4.5, 1)),
 (2020, (3.5, 1)),
 (2020, (4.0, 1)),
 (2020, (3.0, 1)),
 (2020, (4.5, 1))]

In [22]:
#Reduce tinh avg rating va tong rating theo nam
ratingsReduced = ratingsNew.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
#Tinh avg
ratingFinal = ratingsReduced.mapValues(lambda x: (x[0] / x[1], x[1]))
ratingFinal.take(5)

[(2020, (3.7527173913043477, 184))]

In [23]:
def format_result(record):
    year = record[0]
    data = record[1]
    avg_rating = data[0]
    total_ratings = data[1]
    return f"{year} - TotalRatings: {total_ratings}, AverageRating: {avg_rating:.2f}"

formatted_results = ratingFinal.map(format_result)
formatted_results.collect()

['2020 - TotalRatings: 184, AverageRating: 3.75']